# Lecture 5: Multi-Agent Patterns (hands-on)
**Strands Agents with Ollama · Lecture 5 of 5** · companion to `Lecture05_Multi_Agent_Patterns.pdf`

You will build all three patterns locally:
1. **Agents as tools**: an orchestrator delegates to specialists
2. **Graph**: a deterministic pipeline with parallel fan-out
3. **Swarm**: autonomous handoffs with safety limits

> Multi-agent runs make several model calls, so they are slow on CPU. A Colab GPU runtime helps a lot.

## 0. Setup: Ollama + Strands
Same steps as `StrandsAgent_SetUp.ipynb`. Run these once per Colab session (about 2 to 3 minutes).

> **Tip:** *Runtime → Change runtime type → T4 GPU* makes Ollama much faster. CPU works too, just slower.

In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq zstd
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background and wait until it answers
import subprocess, time, urllib.request
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start - check /tmp/ollama.log")

In [ ]:
!ollama pull qwen3:0.6b
# Optional, more reliable for tool use (1.4 GB):
# !ollama pull qwen3:1.7b

In [ ]:
!pip install -q "strands-agents[ollama,web-fetch]"

### Shared model helper
Every lecture uses the same `make_model()` helper. Change `MODEL_ID` here to switch models for the whole notebook.

In [ ]:
from strands.models.ollama import OllamaModel

MODEL_ID = "qwen3:0.6b"          # try "qwen3:1.7b" for better tool use

def make_model(model_id=MODEL_ID, **kwargs):
    return OllamaModel(
        host="http://127.0.0.1:11434",
        model_id=model_id,
        options={"num_ctx": 8192},   # explicit context window (see Lecture 4)
        **kwargs,
    )

import importlib.metadata as md_
print("strands-agents", md_.version("strands-agents"), "| model:", MODEL_ID)

## 1. Tools for this lab

In [ ]:
from strands import Agent, tool

@tool
def query_product_database(query: str) -> str:
    """Query the internal product database for inventory and pricing information.

    Args:
        query: Search query for products (e.g., "wireless headphones", "USB-C hub")
    """
    products = {
        "wireless headphones": "SKU-WH100: Wireless Headphones Pro - $79.99, 142 in stock, 4.5★ rating, launched 2025-03",
        "usb-c hub": "SKU-UC200: USB-C Hub 7-in-1 - $45.00, 89 in stock, 4.2★ rating, launched 2024-11",
        "mechanical keyboard": "SKU-MK300: Mechanical Keyboard RGB - $149.99, 23 in stock, 4.8★ rating, launched 2025-01",
        "noise cancelling": "SKU-NC400: Noise Cancelling Earbuds - $129.99, 67 in stock, 4.6★ rating, launched 2025-05",
    }
    key = query.lower()
    matches = [info for product_key, info in products.items() if product_key in key]
    if matches:
        return "\n".join(matches)
    return f"No products found matching '{query}'. Available: wireless headphones, usb-c hub, mechanical keyboard, noise cancelling"

@tool
def check_competitor_price(product: str) -> str:
    """Look up competitor prices for a product category.

    Args:
        product: Product category, e.g. "wireless headphones"
    """
    data = {"wireless headphones": "SoundMax $89.99, AudioPeak $74.50, BeatLine $99.00"}
    return data.get(product.lower(), f"No competitor data for '{product}'")

## 2. Agents as tools: pass the agent directly
Strands wraps the agent automatically using its `name` and `description`. **The description is the routing logic.**

In [ ]:
catalog = Agent(
    name="catalog_specialist",
    description="Looks up our internal product prices and stock levels.",
    model=make_model(),
    tools=[query_product_database],
    system_prompt="Use query_product_database and report the result briefly.",
    callback_handler=None,
)

manager = Agent(
    model=make_model(),
    tools=[catalog],
    system_prompt="You are a sales manager. Delegate product lookups to catalog_specialist.",
)
result = manager("How much do our wireless headphones cost?")

In [ ]:
# Which "tools" did the manager call?
print([b["toolUse"]["name"] for m in manager.messages for b in m["content"] if "toolUse" in b])
# The specialist kept its own, separate context:
print(len(catalog.messages), "messages in the specialist's context")

## 3. Agents as tools: wrap with `@tool`
This gives full control: a fresh specialist per call (clean context), its own model, and post-processing of the result.

In [ ]:
@tool
def pricing_specialist(product: str) -> str:
    """Compare our price for a product against competitors and return a 2-line verdict.

    Args:
        product: Product category, e.g. "wireless headphones"
    """
    specialist = Agent(
        model=make_model(),              # could be a different, cheaper model
        tools=[query_product_database, check_competitor_price],
        system_prompt="Use both tools, then give a 2-line verdict: are we cheaper or more expensive?",
        callback_handler=None,
    )
    return str(specialist(f"Compare pricing for {product}"))

manager = Agent(
    model=make_model(),                  # try make_model("qwen3:1.7b") here
    tools=[pricing_specialist],
    system_prompt="You are a sales manager. Use pricing_specialist for any pricing comparison.",
)
manager("Are our wireless headphones priced competitively?")

## 4. Graph: deterministic pipeline with fan-out
`research` → (`analysis` ∥ `summarize`) → `report`. The report waits for both branches.

In [ ]:
from strands.multiagent import GraphBuilder

def specialist(name, prompt):
    return Agent(name=name, model=make_model(), system_prompt=prompt, callback_handler=None)

researcher = specialist("researcher", "List 4 short facts about the topic.")
analyst    = specialist("analyst", "Identify 2 trends or insights from the research.")
summarizer = specialist("summarizer", "Condense the research into 3 bullet points.")
writer     = specialist("report_writer", "Write a 1-paragraph report from the analysis and summary.")

builder = GraphBuilder()
builder.add_node(researcher, "research")
builder.add_node(analyst, "analysis")
builder.add_node(summarizer, "summarize")
builder.add_node(writer, "report")
builder.add_edge("research", "analysis")
builder.add_edge("research", "summarize")
builder.add_edge("analysis", "report")
builder.add_edge("summarize", "report")
builder.set_execution_timeout(900)

graph = builder.build()
result = graph("Why are local LLMs useful for developers?")

In [ ]:
print(result.status)
print("order:", [n.node_id for n in result.execution_order])
for node_id, node_result in result.results.items():
    print(f"\n--- {node_id} ---")
    print(str(node_result.result).strip()[:400])

## 5. Swarm: autonomous handoffs
Strands injects a `handoff_to_agent` tool automatically. **Always set limits.**

> In testing, triage handed off correctly but also answered itself despite the prompt. Check `node_history` **and** each node's result.

In [ ]:
from strands.multiagent import Swarm

def member(name, prompt):
    return Agent(name=name, model=make_model(), system_prompt=prompt, callback_handler=None)

triage = member("triage", "You triage questions. For math questions hand off to math_expert. "
                          "For writing questions hand off to writer. Never answer yourself.")
math_expert = member("math_expert", "You solve math questions step by step and give the final answer.")
writer = member("writer", "You write short, clear text.")

swarm = Swarm(
    [triage, math_expert, writer],
    entry_point=triage,
    max_handoffs=4,
    max_iterations=6,
    execution_timeout=600.0,
    node_timeout=180.0,
    repetitive_handoff_detection_window=6,
    repetitive_handoff_min_unique_agents=2,
)
result = swarm("What is 12 times 12?")
print(result.status, [n.node_id for n in result.node_history])
for name, r in result.results.items():
    print(f"\n--- {name} ---\n{str(r.result).strip()[:300]}")

In [ ]:
result = swarm("Write a two-line poem about local AI models.")
print(result.status, [n.node_id for n in result.node_history])

## 6. Your turn
**Exercise 1:** Make the `description` of `catalog_specialist` vague (e.g. "Helps."). Does the manager still route correctly?

**Exercise 2:** Use `make_model("qwen3:1.7b")` for the orchestrator only (section 3) and compare its routing with the all-0.6b version.

**Capstone (stretch):** Build a support desk as a Graph: `triage` → a Swarm of `billing` and `tech` agents → `reply_writer`. Add Lecture 3's `RefundPolicyHook` to the billing agent and a Lecture 4 session manager to the whole thing.

In [ ]:
# Capstone here